# DISTRIBUTION-AWARE V1 — CANDIDATE-SIDE DIAGNOSTIC

**Select a T4 GPU, then Runtime -> Run all.** Roughly one hour. It writes to
Drive under `results/distribution_aware_diagnostic/` and **touches nothing
else**.

This is the cheap diagnostic frozen by
`docs/distribution_aware_decision_memo_2026-09-06.md`. It decides whether
`distribution_aware_v1` earns a detector run at all.

* **No PROB training. No PROB evaluation. No checkpoint is written.** The only
  detector call is `predict`, which scores candidate images. A test asserts the
  driver cannot reach `bridge.train` or `bridge.evaluate`.
* **Nothing of the frozen benchmark is touched.** Different results directory,
  different workspaces. Seeds 0, 1 and 2 of `random`, `admissibility`,
  `entropy`, `proposed`, `proposed_v2` are not read and not written.
* **The gates are printed before anything is measured**, as machine-readable
  JSON, so the criteria are on the record ahead of the outcome. Once the
  outcome is visible none of them may change: not a threshold, not an
  aggregation, not the clusterer, not `min_cluster_size`, not `R`.
* **One declared deviation.** A real trajectory scores task *n*'s pool with the
  checkpoint task *n-1* produced for that arm. This trains nothing, so every
  task of every arm is scored with the **t1 anchor**. The detector is held
  fixed and the comparison becomes purely one about selection — which is what a
  candidate-side diagnostic is for — but `entropy` here is entropy under the
  anchor, not under an arm's own evolving model. It applies identically to all
  three arms.
* `distribution_aware_v1` and `cost_aware` are **development-seed-informed and
  not pre-registered**. Any table reporting them must say so.

**It is resumable in the cheapest sense**: `predict` and the DINOv2 export are
both cached on their output paths, so a disconnected session re-runs the
selection arithmetic and not the GPU work.

In [ ]:
# [1/7] Parameters and immutable experiment identity
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "e9fc9401c0de6d4b88e605c4760086c8437a5405"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
RESULTS_RELATIVE = "results/distribution_aware_diagnostic"
FEATURES_RELATIVE = "features"
DATA_ROOT = "/content/data/OWOD"

# The seeds the memo freezes the gates at. Everything else scientific is read
# from the pinned modules, never retyped here: a number in a notebook cell is a
# number that can drift away from the one the gates are applied with.
SESSION_SEEDS = (0, 1)

assert len(PROB_COMMIT) == 40 and len(OWL_COMMIT) == 40, "pin full 40-char SHAs"
print("OWL commit :", OWL_COMMIT)
print("PROB commit:", PROB_COMMIT)
SESSION_STARTED = time.monotonic()


In [ ]:
# [2/7] Mount Drive, prove the root is writable, and name what stays untouched
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_probe = DRIVE / ".diagnostic_write_probe"
_probe.write_text("ok", encoding="utf-8")
assert _probe.read_text(encoding="utf-8") == "ok"
_probe.unlink()

RESULTS = DRIVE / RESULTS_RELATIVE
FEATURES = DRIVE / FEATURES_RELATIVE
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
RESULTS.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT.exists(), f"the t1 anchor is not at {CHECKPOINT}"

FROZEN = DRIVE / "results" / "full_owod_active_benchmark_v1"
assert RESULTS != FROZEN and FROZEN not in RESULTS.parents
print("writes to :", RESULTS)
print("untouched :", FROZEN, "(exists)" if FROZEN.exists() else "(absent)")


In [ ]:
# [3/7] Pin OWL exactly, install its dependencies, import fresh code
ROOT = Path("/content/owod-active")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--quiet", OWL_REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "fetch", "--quiet", "origin", OWL_COMMIT],
               check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--quiet", OWL_COMMIT], check=True)
_head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"],
                       check=True, capture_output=True, text=True).stdout.strip()
assert _head == OWL_COMMIT, f"checked out {_head}, wanted {OWL_COMMIT}"

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(ROOT)],
               check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "scikit-learn>=1.3"], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from owl import bridge
from owl.active_selection import arms, diagnostic
from owl.active_selection import benchmark as bm

print("owl imported from", ROOT, "at", _head)
print("registered arms:", sorted(arms.ARMS))


In [ ]:
# [4/7] THE FROZEN GATES — printed before anything is measured
if "diagnostic" not in globals():
    raise RuntimeError(
        "cell [3/7] has not run in this kernel, so `diagnostic` does not exist. "
        "It is the cell that pins OWL and imports the package. Use "
        "Runtime -> Run all rather than running cells individually.")

print(json.dumps(diagnostic.configuration(), indent=2))
print()
print("PROVENANCE — this session runs arms that are NOT pre-registered:")
for _line in bm.PROVENANCE:
    print("  *", _line)


In [ ]:
# [5/7] PROB and the data root
PROB_ROOT = Path("/content/PROB")
if not PROB_ROOT.exists():
    subprocess.run(["git", "clone", "--quiet", PROB_REPOSITORY, str(PROB_ROOT)],
                   check=True)
subprocess.run(["git", "-C", str(PROB_ROOT), "checkout", "--quiet", PROB_COMMIT],
               check=True)
subprocess.run([sys.executable, str(ROOT / "tools" / "prepare_full_owod_benchmark.py"),
                "--data-root", DATA_ROOT, "--prob-root", str(PROB_ROOT),
                "--staging", str(ROOT / "data" / "staging")], check=True)

# The balanced task-1 reference. `distribution_aware_v1` measures
# under-representation against it plus everything the trajectory has bought, so
# without it the reference at t2 is empty and every cluster reads as maximally
# under-represented. That is a defined behaviour, not a crash — but it is a
# different measurement, so the run says which one it made.
REF_T1 = FEATURES / "ref_t1_dinov2_vitb14_cap1000.npz"
print("ref-t1 export:", REF_T1, "PRESENT" if REF_T1.exists() else "ABSENT")


In [ ]:
# [6/7] Run the diagnostic. THIS IS THE LONG CELL — about one hour.
_command = [
    sys.executable, str(ROOT / "tools" / "run_distribution_aware_diagnostic.py"),
    "--prob-root", str(PROB_ROOT),
    "--data-root", DATA_ROOT,
    "--checkpoint", str(CHECKPOINT),
    "--out", str(RESULTS),
    "--seeds", *[str(s) for s in SESSION_SEEDS],
]
if REF_T1.exists():
    _command += ["--ref-t1", str(REF_T1)]
print(" ".join(_command), flush=True)
subprocess.run(_command, check=True)


In [ ]:
# [7/7] The verdict, read back from what was written
_verdict = json.loads((RESULTS / "verdict.json").read_text(encoding="utf-8"))
for _gate in _verdict["gates"]:
    print(f"{_gate['verdict']:>4}  {_gate['gate']:<32} "
          f"threshold {_gate['threshold']:<6} measured {_gate['measured_per_seed']}")
print()
print("VERDICT:", _verdict["verdict"])
if _verdict["failed"]:
    print("failed :", ", ".join(_verdict["failed"]))
    print("\nThe memo's rule: preserve as a negative candidate-side result. Do "
          "not train, do not tune, do not re-choose the clusterer, do not "
          "soften a threshold. Report which criterion failed and by how much.")
else:
    print("\nGO. Freeze the method, then run the minimal downstream experiment "
          "— distribution_aware_v1 AND cost_aware, seed 0, ~4.5 T4-hours. Do "
          "not launch it from this notebook.")
print(f"\nelapsed {(time.monotonic() - SESSION_STARTED) / 60:.1f} min")
print("rows :", RESULTS / "diagnostic_rows.csv")
